# FastAPI 기초
* FastAPI를 구성하는 핵심 개념을 **하나씩 만들고 출력하며** 이해합니다
* 15.1에서 State·Node·Graph를 하나씩 찍어본 것과 같은 방식으로 진행합니다
* (프로젝트 준비) 18·19일 실행형 서비스를 위한 FastAPI 기초입니다.
* Day 17 Advanced RAG 이론 실습은 `17.1`을 보세요.



---
* 새로운 Conda 환경에서 실습을 진행하겠습니다.

  아래 명령어로 Python 3.12 환경(`day17`)을 생성한 후, 해당 환경을 활성화하고 ipykernel을 설치합니다
  
  ```
  conda create -n day17 python=3.12
  conda activate day17
  conda install ipykernel
  ```
  
  이렇게 하면 Jupyter Notebook에서 'day17' 환경을 커널로 사용할 수 있습니다.
* 이미 `day15` 환경이 있다면 그 커널을 써도 됩니다. (`fastapi`, `uvicorn`만 추가 설치)


In [ ]:
!pip install fastapi uvicorn python-dotenv httpx


In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)


---
## FastAPI로 API 구현하기


---
## `FastAPI` 앱 객체 만들기

FastAPI에서는 **`FastAPI()`** 로 앱을 만듭니다


In [ ]:
from fastapi import FastAPI

app = FastAPI(title='Day17 Demo')

---
## 첫 엔드포인트 — `@app.get`

* `@app.get('/hello')` 데코레이터 = "GET /hello 요청이 오면 이 함수를 실행하라"
* 함수가 반환한 `dict` / `list` 는 자동으로 **JSON** 응답이 됩니다


In [ ]:
@app.get('/hello')
def hello():
    return {'message': '안녕하세요'}

print('직접 호출:', hello())

---
## 노트북에서 요청 보내기 — `TestClient`

* 실제 서버(`uvicorn`)를 켜지 않고도, **같은 프로세스 안에서** HTTP 요청을 흉내 낼 수 있습니다
* 오늘 실습의 기본 도구입니다


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
client


In [ ]:
response = client.get('/hello')

print(response.status_code)
print(response.json())

* `status_code == 200` → 성공
* `response.json()` → 본문을 파이썬 `dict`로 파싱


In [ ]:
# 없는 경로를 호출하면?
bad = client.get('/없는경로')
print(bad.status_code)
print(bad.json())

---
## Path Parameter (경로 변수)

* URL 경로에 값이 들어갑니다: `/items/3` 의 `3`
* 함수 인자 이름과 `{item_id}` 이름을 **같게** 맞춥니다


In [ ]:
@app.get('/items/{item_id}')
def read_item(item_id: int):
    return {'item_id': item_id}

In [ ]:
r = client.get('/items/42')
print(r.status_code, r.json())


In [ ]:
# 타입이 int 인데 문자열을 넣으면?
r2 = client.get('/items/abc')
print(r2.status_code)
print(r2.json())  # FastAPI가 자동으로 422 Validation Error

* `item_id: int` 타입 힌트 덕분에 FastAPI가 **자동 검증·변환**을 합니다
* 15.1의 Pydantic 검증과 같은 철학입니다


---
## Query Parameter (쿼리 변수)

* URL의 `?` 뒤에 붙습니다: `/search?q=연세&limit=5`
* 경로에 `{ }`가 없고, 함수 인자로 선언하면 Query로 취급됩니다


In [ ]:
@app.get('/search')
def search(q: str, limit: int = 10):
    return {'q': q, 'limit': limit, 'results': [f'{q}-{i}' for i in range(limit)]}

print(search(q='학칙', limit=3))

In [ ]:
r = client.get('/search', params={'q': '학칙', 'limit': 2})
print('요청 URL :', r.request.url)
print('응답     :', r.json())


---
## HTTP 메서드 여러 개

| 메서드 | 데코레이터 | 관례적 용도 |
|--------|------------|-------------|
| GET | `@app.get` | 조회 |
| POST | `@app.post` | 생성·제출 |
| PUT | `@app.put` | 전체 수정 |
| DELETE | `@app.delete` | 삭제 |

* 같은 경로라도 메서드가 다르면 **다른 엔드포인트**입니다


In [ ]:
NOTES = []  # 간단한 메모리 저장소

@app.get('/notes')
def list_notes():
    return {'notes': NOTES, 'count': len(NOTES)}

@app.post('/notes')
def add_note(text: str):
    NOTES.append(text)
    return {'ok': True, 'notes': NOTES}


In [ ]:
print(client.get('/notes').json())
print(client.post('/notes', params={'text': '첫 메모'}).json())
print(client.post('/notes', params={'text': '둘째 메모'}).json())
print(client.get('/notes').json())


---
## 실제로 서버 띄우기

노트북 밖 터미널에서:

```bash
cd "17일차_실습"
uvicorn hello:app --reload
```

* `hello` = `hello.py` 파일 이름
* `app` = 그 안의 `FastAPI()` 변수 이름
* 브라우저에서 `http://127.0.0.1:8000/hello` 접속